# Silver Layer — Imperative Approach

## Objective

Transform the Bronze Ingestion Delta table into two clean, typed and 
validated Silver tables ready for analytical consumption.

This notebook implements the **imperative approach** — every step 
(read, cast, deduplicate, validate, write) is explicit. A declarative 
DLT version is implemented in `04_silver_declarative`.

## Architecture

# Silver Layer — Imperative Approach

## Objective

Transform the Bronze Ingestion Delta table into two clean, typed and 
validated Silver tables ready for analytical consumption.

This notebook implements the **imperative approach** — every step 
(read, cast, deduplicate, validate, write) is explicit. A declarative 
DLT version is implemented in `04_silver_declarative`.

## Notebook Flow

1. **Read source**  
   Load the Delta table from Bronze Ingestion.

2. **Build `silver_dim_stock`**
   - Select dimension attributes (`symbol`, `last_refreshed`, `time_zone`)
   - Cast `last_refreshed` to DATE
   - Deduplicate by `symbol`
   - Add `ingest_timestamp`

3. **Validate `silver_dim_stock`**
   - Drop rows where `symbol`, `last_refreshed` or `time_zone` are NULL
   - Log the number of dropped rows

4. **Build `silver_fact_prices`**
   - Select fact attributes (`symbol`, `trade_date`, OHLCV)
   - Cast `trade_date` to DATE
   - Cast `open`, `high`, `low`, `close` to DOUBLE
   - Cast `volume` to LONG
   - Deduplicate by composite key (`symbol`, `trade_date`)
   - Add `ingest_timestamp`

5. **Validate `silver_fact_prices`**
   - Apply Data Quality rules: `open > 0`, `high > 0`, `low > 0`, 
     `close > 0`, `volume >= 0`
   - Log the number of dropped rows

6. **Write to ADLS Silver**  
   Persist both tables as Delta with `overwrite` mode using a generic 
   write function that supports optional partitioning.

## Design Decisions

- **Type casting** is performed in Silver (not Bronze) to keep raw 
  data immutable in Bronze Landing
- **Deduplication** uses natural keys — no surrogate keys
- **Data Quality** is enforced by filtering invalid rows; the count 
  of dropped rows is logged
- **Overwrite mode** assumes Silver is recomputed from Bronze on 
  each run, ensuring idempotency
- **Write function is generic** — accepts mode and partition_by 
  parameters for reuse in Gold layer

In [0]:
BRONZE_INGESTION_PATH = "abfss://bronze@marketpulsedatalake.dfs.core.windows.net/ingestion/stocks/"

df_bronze = spark.read.format('delta').load(BRONZE_INGESTION_PATH)
df_bronze.printSchema()
df_bronze.show(5)


In [0]:

# imports and configuration
import json
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, to_date, current_timestamp

# Paths — single source of truth
STORAGE_ACCOUNT = "marketpulsedatalake"
BRONZE_INGESTION_PATH = f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net/ingestion/stocks/"
SILVER_DIM_STOCK_PATH = f"abfss://silver@{STORAGE_ACCOUNT}.dfs.core.windows.net/dim_stock/"
SILVER_FACT_PRICES_PATH = f"abfss://silver@{STORAGE_ACCOUNT}.dfs.core.windows.net/fact_prices/"

print("✅ Configuration loaded")
print(f"  Source:      {BRONZE_INGESTION_PATH}")
print(f"  Dim target:  {SILVER_DIM_STOCK_PATH}")
print(f"  Fact target: {SILVER_FACT_PRICES_PATH}")


In [0]:
#read function

def read_bronze() -> DataFrame:
    """
    Reads the Bronze Ingestion Delta table.
    
    Returns:
        DataFrame with all flattened stock records (all columns as STRING).
    """    
    
    return spark.read.format('delta').load(BRONZE_INGESTION_PATH)

In [0]:
#read function
def read_bronze_ingestion() -> DataFrame:
    
    return spark.read.format("delta").load(BRONZE_INGESTION_PATH)


df_bronze = read_bronze_ingestion()
print(f"✅ Bronze Ingestion loaded — {df_bronze.count()} rows")

In [0]:
def build_dim_stock(df_bronze: DataFrame) -> DataFrame:
    """
    Builds the Silver dimension table for stocks.
        
    Returns:
        DataFrame with columns: symbol, last_refreshed (DATE),
        time_zone, ingest_timestamp.
    """
    return(
        df_bronze.select("symbol","last_refreshed", "time_zone")
        .withColumn("last_refreshed", to_date(col("last_refreshed")))
        .withColumn("ingest_timestamp", current_timestamp())
        .dropDuplicates(["symbol"])
    )



In [0]:
df_dim = build_dim_stock(df_bronze)
df_dim.printSchema()
df_dim.show()

In [0]:
def validate_dim_stock(df_dim: DataFrame) -> DataFrame:
    """
    Applies data quality rules to the dim stock DataFrame.
    Drops invalid rows and logs the count of dropped records.
    """
    total_before = df_dim.count() # before validating
    df_valid = df_dim.filter((col("symbol").isNotNull()) & (col("last_refreshed").isNotNull()) & (col("time_zone").isNotNull()))
    total_after = df_valid.count()
    dropped = total_before - total_after
    print(f"  Dropped {dropped} invalid records")
    return df_valid




In [0]:
df_dim_validated = validate_dim_stock(df_dim)
print(f"\nFinal valid rows: {df_dim_validated.count()}")

In [0]:
def build_fact_prices(df_bronze: DataFrame) -> DataFrame:
    """
    Builds the Silver fact table for stock prices.
        
    Returns:
        DataFrame with columns: symbol, trade_date (DATE), open, high, low, close, volume.
    """
    return (
        df_bronze.select("symbol","trade_date", "open", "high", "low", "close", "volume")
        .withColumn("trade_date", to_date(col("trade_date")))
        .withColumn("open", col("open").cast("double"))
        .withColumn("high", col("high").cast("double"))
        .withColumn("low", col("low").cast("double"))
        .withColumn("close", col("close").cast("double"))
        .withColumn("volume", col("volume").cast("long"))
        .withColumn("ingest_timestamp", current_timestamp())
        .dropDuplicates(["symbol","trade_date"])
    )

In [0]:
df_fact_prices = build_fact_prices(df_bronze)
df_fact_prices.printSchema()
df_fact_prices.show(5)

In [0]:
def validate_fact_prices(df_fact_prices: DataFrame) -> DataFrame:
    """
    Applies data quality rules to the fact prices DataFrame.
    Drops invalid rows and logs the count of dropped records.
    """
    total_before = df_fact_prices.count() # before validating
    df_valid = df_fact_prices.filter((col("open") > 0) & (col("high") > 0) & (col("low") > 0) & (col("close") > 0) & (col("volume") >= 0))
    total_after = df_valid.count()
    dropped = total_before - total_after
    print(f"  Dropped {dropped} invalid records")
    return df_valid



In [0]:
df_fact_validated = validate_fact_prices(df_fact_prices)
print(f"\nFinal valid rows: {df_fact_validated.count()}")

In [0]:
def write_silver_table(df: DataFrame, path:str, table_name:str, mode:str="overwrite", partition_by: list = None) -> None:
    """ Writes a DataFrame as a Delta table to the Silver layer.
    
    Args:
        df:           DataFrame to write
        path:         ADLS destination path
        table_name:   Logical name (used in logs)
        mode:         Write mode (overwrite, append, ...)
        partition_by: Optional list of columns to partition by ready for the ones who will need it
    """
    writer = df.write.format("delta").mode(mode)
    if partition_by is not None:
        writer = writer.partitionBy(partition_by)
    writer.save(path)
    print(f"  Wrote {table_name} to {path}")
write_silver_table(df_fact_validated, SILVER_FACT_PRICES_PATH, "silver_prices", "overwrite")
write_silver_table(df_dim_validated, SILVER_DIM_STOCK_PATH, "silver_stocks", "overwrite")

In [0]:
%sql
-- how many single days for stock??
SELECT 
    symbol,
    COUNT(*) as days,
    MIN(trade_date) as first_day,
    MAX(trade_date) as last_day
FROM delta.`abfss://silver@marketpulsedatalake.dfs.core.windows.net/fact_prices/`
GROUP BY symbol
ORDER BY symbol